In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

def get_pose_anchor_2d(df, side):
    """Lấy tọa độ trung bình 2D (x, y) của Wrist, Pinky, Index"""
    names = [f'{side.upper()}_WRIST', f'{side.upper()}_PINKY', f'{side.upper()}_INDEX']
    points = []
    for name in names:
        row = df[df['landmark_name'] == name]
        if not row.empty:
            # CHỈ lấy x và y để so sánh side
            v = pd.to_numeric(row[['x', 'y']].values[0], errors='coerce')
            if not np.isnan(v).any():
                points.append(v)
    if not points: return None
    return np.mean(points, axis=0)

def clean_and_relabel_hands(df):
    # 1. Lấy mốc Pose chuẩn 2D (x, y)
    pose_l = get_pose_anchor_2d(df, 'LEFT')
    pose_r = get_pose_anchor_2d(df, 'RIGHT')
    
    # Bước 1: Thu thập tất cả cụm tay thực sự có tọa độ
    all_hands = df[df['type'].str.contains('hand', na=False)].copy()
    all_hands['x_num'] = pd.to_numeric(all_hands['x'], errors='coerce')
    real_hands = all_hands.dropna(subset=['x_num'])
    
    # Tách thành từng cụm 21 điểm
    chunks = [real_hands.iloc[i:i+21] for i in range(0, len(real_hands), 21)]
    
    best_left = None
    best_right = None

    # Bước 2: Logic Bắt cặp tối ưu dựa trên khoảng cách 2D
    num_chunks = len(chunks)

    if num_chunks == 1:
        # Nếu chỉ có 1 cụm, tính dist 2D tới 2 mốc Pose
        h0_2d = pd.to_numeric(chunks[0][chunks[0]['landmark_name'] == 'HAND_0'][['x', 'y']].values[0], errors='coerce')
        dist_l = np.linalg.norm(h0_2d - pose_l) if pose_l is not None else 999
        dist_r = np.linalg.norm(h0_2d - pose_r) if pose_r is not None else 999
        
        # Chỉ chuyển nhãn nếu khoảng cách tới bên kia thực sự gần hơn (có sai số an toàn)
        if dist_l < dist_r:
            best_left = chunks[0].copy()
        else:
            best_right = chunks[0].copy()

    elif num_chunks >= 2:
        # So sánh 2 phương án ghép cặp tối ưu (PA A vs PA B) bằng khoảng cách 2D
        c1, c2 = chunks[0], chunks[1]
        h0_c1_2d = pd.to_numeric(c1[c1['landmark_name'] == 'HAND_0'][['x', 'y']].values[0], errors='coerce')
        h0_c2_2d = pd.to_numeric(c2[c2['landmark_name'] == 'HAND_0'][['x', 'y']].values[0], errors='coerce')

        # PA A: C1=Left, C2=Right
        dist_A = (np.linalg.norm(h0_c1_2d - pose_l) if pose_l is not None else 0) + \
                 (np.linalg.norm(h0_c2_2d - pose_r) if pose_r is not None else 0)
        
        # PA B: C1=Right, C2=Left
        dist_B = (np.linalg.norm(h0_c1_2d - pose_r) if pose_r is not None else 0) + \
                 (np.linalg.norm(h0_c2_2d - pose_l) if pose_l is not None else 0)

        if dist_A < dist_B:
            best_left, best_right = c1.copy(), c2.copy()
        else:
            best_left, best_right = c2.copy(), c1.copy()

    # Bước 3: Tái cấu trúc DataFrame (Giữ nguyên cấu trúc 543 dòng)
    df_no_hands = df[~df['type'].str.contains('hand', na=False)].copy()
    
    def create_hand_df(chunk, side):
        if chunk is not None:
            res = chunk.copy().drop(columns=['x_num'])
            res['type'] = f'hand_{side}'
            return res
        else:
            return pd.DataFrame([{
                'landmark_name': f'HAND_{i}', 'type': f'hand_{side}', 
                'x': '', 'y': '', 'z': '', 'visibility': ''
            } for i in range(21)])

    final_left = create_hand_df(best_left, 'Left')
    final_right = create_hand_df(best_right, 'Right')

    result_df = pd.concat([df_no_hands, final_left, final_right], ignore_index=True)
    return result_df[['landmark_name', 'type', 'x', 'y', 'z', 'visibility']]

# --- ĐƯỜNG DẪN (Tùy chỉnh theo máy của bạn) ---
input_dir = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords_2")
output_dir = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords_preprocess_2")

for split in ['train', 'dev', 'test']:
    path_split = input_dir / split
    if not path_split.exists(): continue
    for sub in tqdm(list(path_split.iterdir()), desc=f"Relabeling {split}"):
        if not sub.is_dir(): continue
        csv_files = sorted(list(sub.glob("*.csv")))
        target_sub_dir = output_dir / split / sub.name
        target_sub_dir.mkdir(parents=True, exist_ok=True)
        for f in csv_files:
            df = pd.read_csv(f)
            cleaned_df = clean_and_relabel_hands(df)
            cleaned_df.to_csv(target_sub_dir / f.name, index=False)

Relabeling test: 100%|██████████| 642/642 [1:15:49<00:00,  7.09s/it]


## nội suy

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

def get_xyz(df, name):
    """Lấy tọa độ [x, y, z] từ landmark_name"""
    row = df[df['landmark_name'] == name]
    if row.empty: return None
    v = pd.to_numeric(row[['x', 'y', 'z']].values[0], errors='coerce')
    return None if np.isnan(v).any() else v

def interpolate_hand_sequence(df_list, side):
    """Logic nội suy bám đuổi Pose (Giữ nguyên như cũ)"""
    hand_type = f'hand_{side}'
    wrist_name = f'{side.upper()}_WRIST'
    
    has_hand = []
    for df in df_list:
        h_rows = df[df['type'] == hand_type]
        x_vals = pd.to_numeric(h_rows['x'], errors='coerce')
        has_hand.append(not x_vals.isna().all())

    indices = [i for i, exists in enumerate(has_hand) if exists]
    if len(indices) < 2: return

    for start_idx, end_idx in zip(indices[:-1], indices[1:]):
        gap_size = end_idx - start_idx
        if gap_size <= 1: continue 
        
        for t in range(start_idx + 1, end_idx):
            if has_hand[t]: continue
            
            alpha = (t - start_idx) / gap_size
            df_s, df_e, df_t = df_list[start_idx], df_list[end_idx], df_list[t]

            p_s, p_e, p_t = get_xyz(df_s, wrist_name), get_xyz(df_e, wrist_name), get_xyz(df_t, wrist_name)

            offset = np.zeros(3)
            if p_s is not None and p_e is not None and p_t is not None:
                p_interp = (1 - alpha) * p_s + alpha * p_e
                offset = p_t - p_interp

            for i in range(21):
                lm_name = f"HAND_{i}"
                try:
                    s_hand = pd.to_numeric(df_s[(df_s['type']==hand_type) & (df_s['landmark_name']==lm_name)][['x','y','z']].values[0], errors='coerce')
                    e_hand = pd.to_numeric(df_e[(df_e['type']==hand_type) & (df_e['landmark_name']==lm_name)][['x','y','z']].values[0], errors='coerce')
                    if np.isnan(s_hand).any() or np.isnan(e_hand).any(): continue

                    final_xyz = ((1 - alpha) * s_hand + alpha * e_hand) + offset
                    mask = (df_t['type'] == hand_type) & (df_t['landmark_name'] == lm_name)
                    df_t.loc[mask, ['x', 'y', 'z']] = final_xyz
                except: continue

# --- QUY TRÌNH THỰC THI (Cập nhật Resume logic) ---
input_dir = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords_preprocess_2")
output_dir = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords_final_2")

for split in ['train', 'dev', 'test']:
    path_split = input_dir / split
    if not path_split.exists(): continue
    
    subfolders = [f for f in path_split.iterdir() if f.is_dir()]
    
    # Dùng tqdm để theo dõi tiến trình
    for sub in tqdm(subfolders, desc=f"Temporal Interpolation {split}"):
        
        # --- LOGIC CHẠY TIẾP (RESUME) ---
        target_sub_dir = output_dir / split / sub.name
        # Nếu thư mục đích đã tồn tại, ta coi như folder này đã xử lý xong và bỏ qua
        if target_sub_dir.exists():
            continue
        # -------------------------------

        csv_files = sorted(list(sub.glob("*.csv")))
        if not csv_files: continue
        
        df_list = [pd.read_csv(f) for f in csv_files]
        
        interpolate_hand_sequence(df_list, 'Left')
        interpolate_hand_sequence(df_list, 'Right')
        
        # Tạo folder và lưu kết quả
        target_sub_dir.mkdir(parents=True, exist_ok=True)
        for i, df in enumerate(df_list):
            df.to_csv(target_sub_dir / csv_files[i].name, index=False)

print(f"\n--- HOÀN TẤT ---")
print(f"Dữ liệu đã được tiếp tục xử lý tại: {output_dir}")

Temporal Interpolation test: 100%|██████████| 642/642 [1:17:24<00:00,  7.23s/it]


--- HOÀN TẤT ---
Dữ liệu đã được tiếp tục xử lý tại: C:\Users\TanHD6\Documents\DATN\phoenix-coords_final_2


In [1]:
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
import mediapipe as mp

# Cấu hình MediaPipe
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose
mp_face = mp.solutions.face_mesh

# 1. THIẾT LẬP ĐƯỜNG DẪN
# Folder chứa CSV sạch
input_folder = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords\train\01April_2010_Thursday_heute-6694")
# Folder để lưu ảnh nền đen
output_folder = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords\train_black\01April_2010_Thursday_heute-6694")

# Tự động tạo folder đầu ra nếu chưa có (bao gồm cả các folder cha)
output_folder.mkdir(parents=True, exist_ok=True)

csv_files = sorted(list(input_folder.glob("*.csv")))

# Kích thước ảnh (Nên để bằng hoặc tỉ lệ với ảnh gốc của PHOENIX)
W, H = 600, 800 

def draw_skeleton(df, width, height):
    # Tạo nền đen
    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    
    def get_pix(row):
        # Kiểm tra dữ liệu trống hoặc NaN
        try:
            val_x = float(row['x'])
            val_y = float(row['y'])
            if np.isnan(val_x) or np.isnan(val_y): return None
            return (int(val_x * width), int(val_y * height))
        except (ValueError, TypeError):
            return None

    # --- VẼ POSE ---
    pose_df = df[df['type'] == 'pose']
    pose_pts = {}
    for _, row in pose_df.iterrows():
        pt = get_pix(row)
        if pt:
            pose_pts[row['landmark_name']] = pt
            cv2.circle(canvas, pt, 3, (0, 255, 255), -1)

    for start_idx, end_idx in mp_pose.POSE_CONNECTIONS:
        s_name = mp_pose.PoseLandmark(start_idx).name
        e_name = mp_pose.PoseLandmark(end_idx).name
        if s_name in pose_pts and e_name in pose_pts:
            cv2.line(canvas, pose_pts[s_name], pose_pts[e_name], (255, 255, 255), 2)

    # --- VẼ HANDS ---
    for side in ['hand_Left', 'hand_Right']:
        hand_df = df[df['type'] == side]
        hand_pts = {}
        color = (0, 255, 0) if 'Left' in side else (255, 0, 0)
        
        for _, row in hand_df.iterrows():
            pt = get_pix(row)
            if pt:
                hand_pts[row['landmark_name']] = pt
                cv2.circle(canvas, pt, 2, color, -1)

        for start_idx, end_idx in mp_hands.HAND_CONNECTIONS:
            s_name, e_name = f"HAND_{start_idx}", f"HAND_{end_idx}"
            if s_name in hand_pts and e_name in hand_pts:
                cv2.line(canvas, hand_pts[s_name], hand_pts[e_name], color, 2)

    # --- VẼ FACE ---
    face_df = df[df['type'] == 'face']
    for _, row in face_df.iterrows():
        pt = get_pix(row)
        if pt:
            cv2.circle(canvas, pt, 1, (255, 0, 255), -1)

    return canvas

# 2. CHẠY VÀ LƯU ẢNH
print(f"Bắt đầu vẽ và lưu {len(csv_files)} ảnh vào: {output_folder}")

for csv_file in csv_files:
    # Đọc dữ liệu CSV
    df = pd.read_csv(csv_file)
    
    # Vẽ skeleton
    frame_vis = draw_skeleton(df, W, H)
    
    # Tạo tên file ảnh (images0001.csv -> images0001.png)
    image_name = csv_file.stem + ".png"
    save_path = output_folder / image_name
    
    # Lưu ảnh xuống ổ đĩa
    cv2.imwrite(str(save_path), frame_vis)
    
    # Hiển thị preview (tùy chọn - có thể tắt để chạy nhanh hơn)
    cv2.imshow("Preview - Saving Black Frames", frame_vis)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()
print("Hoàn tất lưu toàn bộ ảnh!")

Bắt đầu vẽ và lưu 53 ảnh vào: C:\Users\TanHD6\Documents\DATN\phoenix-coords\train_black\01April_2010_Thursday_heute-6694
Hoàn tất lưu toàn bộ ảnh!


## xử lý mặt

In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# --- CẤU HÌNH ---
input_root = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords")
output_root = Path(r"C:\Users\TanHD6\Documents\DATN\phoenix-coords_fixed_final")
DATA_ROWS_CORRECT = 553 # Tương ứng 554 dòng trong Excel (trừ 1 dòng header)
splits = ['train', 'dev', 'test']

def fix_face_landmarks():
    print(f"--- BẮT ĐẦU ĐỒNG BỘ: 553 DÒNG DỮ LIỆU (554 DÒNG EXCEL) ---")
    
    for split in splits:
        split_path = input_root / split
        if not split_path.exists(): continue

        video_folders = [d for d in split_path.iterdir() if d.is_dir()]
        
        for video_dir in tqdm(video_folders, desc=f"Split {split:5}"):
            target_sub_dir = output_root / split / video_dir.name
            csv_files = sorted(list(video_dir.glob("*.csv")))
            if not csv_files: continue

            # Logic Resume: Nếu folder đã đủ file thì bỏ qua
            if target_sub_dir.exists() and len(list(target_sub_dir.glob("*.csv"))) == len(csv_files):
                continue

            target_sub_dir.mkdir(parents=True, exist_ok=True)
            last_valid_face = None

            # 1. Tìm khuôn mặt "mẫu" 478 điểm đầu tiên trong video để làm mốc cứu viện
            for f_check in csv_files:
                temp_df = pd.read_csv(f_check)
                if len(temp_df) == DATA_ROWS_CORRECT:
                    # Lưu lại đúng 478 dòng có type là 'face'
                    last_valid_face = temp_df[temp_df['type'] == 'face'].copy()
                    break

            # 2. Xử lý từng file
            for csv_path in csv_files:
                df = pd.read_csv(csv_path)
                row_count = len(df) # Pandas đếm dòng dữ liệu (không tính header)
                
                # --- TRƯỜNG HỢP A: FILE ĐÃ CHUẨN (553 DÒNG DỮ LIỆU) ---
                if row_count == DATA_ROWS_CORRECT:
                    # Cập nhật mặt mốc mới nhất
                    last_valid_face = df[df['type'] == 'face'].copy()
                    # Lưu nguyên bản sang folder mới
                    df.to_csv(target_sub_dir / csv_path.name, index=False)
                
                # --- TRƯỜNG HỢP B: FILE LỖI (543 DÒNG DỮ LIỆU) ---
                else:
                    if last_valid_face is not None:
                        # Tách lấy Pose và Hand (75 dòng: 33 pose + 42 hands)
                        df_pose_hand = df[df['type'] != 'face'].copy()
                        
                        # Kết hợp với 478 dòng mặt "mượn"
                        fixed_df = pd.concat([df_pose_hand, last_valid_face], ignore_index=True)
                        
                        # Lưu file đã sửa (75 + 478 = 553 dòng dữ liệu)
                        fixed_df.to_csv(target_sub_dir / csv_path.name, index=False)
                    else:
                        # Nếu cả video không có mặt (hiếm), copy nguyên bản để không mất file
                        df.to_csv(target_sub_dir / csv_path.name, index=False)

if __name__ == "__main__":
    fix_face_landmarks()
    print(f"\n--- HOÀN TẤT ---")
    print(f"Tất cả file tại '{output_root}' đã được đưa về chuẩn 553 dòng dữ liệu (554 lines).")

--- BẮT ĐẦU ĐỒNG BỘ: 553 DÒNG DỮ LIỆU (554 DÒNG EXCEL) ---


Split test : 100%|██████████| 642/642 [55:56<00:00,  5.23s/it]  


--- HOÀN TẤT ---
Tất cả file tại 'C:\Users\TanHD6\Documents\DATN\phoenix-coords_fixed_final' đã được đưa về chuẩn 553 dòng dữ liệu (554 lines).
